# Module 13 lab — KV cache from first principles

This notebook is self-contained and runs on one NVIDIA T4. It inspects real Qwen K/V tensors, derives logical storage, verifies one-token growth, measures linear cache growth, and compares cached with uncached generation.

## Lab A — Install and load the model

Use a Transformers 4.x environment. Restart the kernel if you changed an already-imported Transformers installation.

In [ ]:
# Install the dependencies needed by this notebook.
%pip install -U "transformers>=4.45,<5" accelerate safetensors pandas matplotlib

In [ ]:
# Import dependencies and select one CUDA device.
import gc
import statistics
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), 'Select your T4 GPU runtime.'
device = torch.device('cuda:0')
model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
).to(device).eval()

cfg = model.config
print('GPU:', torch.cuda.get_device_name(0))
print('Layers:', cfg.num_hidden_layers)
print('Query heads:', cfg.num_attention_heads)
print('KV heads:', cfg.num_key_value_heads)
print('Head dimension:', cfg.hidden_size // cfg.num_attention_heads)
print('Model dtype:', next(model.parameters()).dtype)

## Lab B — Create and inspect a real KV cache

The prefill pass computes K and V for every prompt position. The cache stores one K tensor and one V tensor per Transformer layer.

In [ ]:
# Use the chat template so the input matches the instruction-tuned model format.
messages = [{
    'role': 'user',
    'content': 'Explain why the KV cache helps autoregressive decoding.',
}]
encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors='pt',
).to(device)

with torch.inference_mode():
    prefill = model(**encoded, use_cache=True, return_dict=True)
cache = prefill.past_key_values
prompt_len = encoded['input_ids'].shape[1]
print('Prompt tokens:', prompt_len)
print('Cache type:', type(cache).__name__)
print('Cached positions:', cache.get_seq_length())

In [ ]:
# Read the first layer using the current DynamicCache API.
if hasattr(cache, 'layers'):
    first_layer = cache.layers[0]
    key = first_layer.keys
    value = first_layer.values
else:
    key, value = cache[0]

print('Number of cache layers:', len(cache.layers) if hasattr(cache, 'layers') else len(cache))
print('K shape:', tuple(key.shape))
print('V shape:', tuple(value.shape))
print('K dtype:', key.dtype)
print('K device:', key.device)
print('Expected layout: [batch, KV heads, cached positions, head dimension]')

For Qwen2.5-0.5B, the expected first-layer layout is `[1, 2, prompt_len, 64]`. There are two KV heads because the model uses grouped-query attention; its 14 query heads share those two K/V groups.

## Lab C — Calculate logical KV-cache storage

The logical storage equation is:

$$M_{\mathrm{KV}}=2BLH_{\mathrm{KV}}Sd_hb$$

This counts K and V tensor contents across every layer. It excludes allocator reservations, temporary buffers, and fragmentation.

In [ ]:
# Sum live logical K/V tensor bytes across all cache layers.
def kv_storage_bytes(cache):
    total = 0
    layers = cache.layers if hasattr(cache, 'layers') else cache
    for layer in layers:
        tensors = (layer.keys, layer.values) if hasattr(layer, 'keys') else layer
        for tensor in tensors:
            total += tensor.numel() * tensor.element_size()
    return total

actual_bytes = kv_storage_bytes(cache)
batch_size = encoded['input_ids'].shape[0]
layers = cfg.num_hidden_layers
kv_heads = cfg.num_key_value_heads
head_dim = cfg.hidden_size // cfg.num_attention_heads
dtype_bytes = key.element_size()
expected_bytes = 2 * batch_size * layers * kv_heads * prompt_len * head_dim * dtype_bytes
print('Measured logical KV bytes:', actual_bytes)
print('Formula prediction:', expected_bytes)
print('Matching:', actual_bytes == expected_bytes)
print('KV storage (MiB):', actual_bytes / 1024**2)
print('Bytes per token:', actual_bytes / prompt_len)

For this configuration, one sequence consumes 12,288 logical FP16 bytes per cached token, or 12 KiB/token. A 4,096-token sequence therefore requires 48 MiB of logical K/V storage.

## Lab D — Process one token and observe cache growth

In [ ]:
# Select the first generated token from the prefill logits.
next_token = prefill.logits[:, -1:, :].argmax(dim=-1)
print('First generated token ID:', next_token.item())
print('First generated token:', repr(tokenizer.decode(next_token[0].tolist())))

old_len = cache.get_seq_length()
old_bytes = kv_storage_bytes(cache)
new_attention_mask = torch.cat(
    [encoded['attention_mask'], encoded['attention_mask'].new_ones((1, 1))],
    dim=-1,
)

with torch.inference_mode():
    decode_output = model(
        input_ids=next_token,
        attention_mask=new_attention_mask,
        past_key_values=cache,
        cache_position=torch.tensor([old_len], dtype=torch.long, device=device),
        use_cache=True,
        return_dict=True,
    )
new_cache = decode_output.past_key_values
new_len = new_cache.get_seq_length()
new_bytes = kv_storage_bytes(new_cache)
print('Old cache length:', old_len)
print('New cache length:', new_len)
print('Additional logical KV bytes:', new_bytes - old_bytes)
assert new_len == old_len + 1
assert new_bytes - old_bytes == 12288

The logical increase should be 12,288 bytes. PyTorch's caching allocator may reserve more memory because a dynamic cache can allocate and copy tensors; logical live tensor bytes and allocator-reserved VRAM are different measurements.

## Lab E — Verify linear cache growth across context lengths

In [ ]:
# Measure logical cache bytes for synthetic prompts of increasing length.
rows = []
for seq_len in [64, 128, 256, 512, 1024, 2048]:
    input_ids = torch.randint(100, 50000, (1, seq_len), device=device)
    with torch.inference_mode():
        output = model(input_ids=input_ids, use_cache=True, return_dict=True)
    measured = kv_storage_bytes(output.past_key_values)
    predicted = 12288 * seq_len
    rows.append({
        'tokens': seq_len,
        'measured_MiB': measured / 1024**2,
        'predicted_MiB': predicted / 1024**2,
        'difference_bytes': measured - predicted,
    })
    del output, input_ids
    gc.collect()

growth_df = pd.DataFrame(rows)
display(growth_df)

In [ ]:
# The measured and predicted curves should overlap for logical tensor storage.
plt.figure(figsize=(8, 4))
plt.plot(growth_df['tokens'], growth_df['measured_MiB'], marker='o', label='Measured')
plt.plot(growth_df['tokens'], growth_df['predicted_MiB'], '--', label='Predicted: 12 KiB/token')
plt.xlabel('Cached positions')
plt.ylabel('Logical KV storage (MiB)')
plt.title('Qwen2.5-0.5B — KV-cache growth')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Lab F — Compare cached and uncached generation

The uncached path recomputes the full accumulated sequence. The cached path processes the prompt once, then supplies only the newest token while reusing prior K/V states.

In [ ]:
# Run fixed-length greedy generation with and without the KV cache.
@torch.inference_mode()
def run_generation(input_ids, steps=8, use_kv_cache=True):
    sequence = input_ids.clone()
    current_input = input_ids
    cache = None
    attention_mask = torch.ones_like(sequence)

    for _ in range(steps):
        cache_position = None
        if use_kv_cache and cache is not None:
            cache_position = torch.tensor(
                [cache.get_seq_length()], dtype=torch.long, device=device
            )
        output = model(
            input_ids=current_input if use_kv_cache else sequence,
            attention_mask=attention_mask,
            past_key_values=cache if use_kv_cache else None,
            cache_position=cache_position,
            use_cache=use_kv_cache,
            return_dict=True,
        )
        token = output.logits[:, -1:, :].argmax(dim=-1)
        sequence = torch.cat([sequence, token], dim=-1)
        attention_mask = torch.cat(
            [attention_mask, attention_mask.new_ones((1, 1))], dim=-1
        )
        if use_kv_cache:
            cache = output.past_key_values
            current_input = token
    return sequence

def benchmark_generation(input_ids, steps=8, use_kv_cache=True, repeats=3):
    # Warm up the selected path before measuring it.
    run_generation(input_ids, 2, use_kv_cache)
    torch.cuda.synchronize()
    times = []
    final_sequence = None
    for _ in range(repeats):
        torch.cuda.synchronize()
        started = time.perf_counter()
        final_sequence = run_generation(input_ids, steps, use_kv_cache)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - started) * 1000)
    return statistics.median(times), final_sequence

benchmark_input = encoded['input_ids']
uncached_ms, uncached_sequence = benchmark_generation(
    benchmark_input, use_kv_cache=False
)
cached_ms, cached_sequence = benchmark_generation(
    benchmark_input, use_kv_cache=True
)
print('Uncached milliseconds:', round(uncached_ms, 2))
print('Cached milliseconds:', round(cached_ms, 2))
print('Speedup:', round(uncached_ms / cached_ms, 2), 'x')
print('Same token IDs:', torch.equal(uncached_sequence, cached_sequence))

The speedup is workload-dependent. Dynamic-cache allocation, context length, kernel selection, and numerical differences can affect the result. The core tradeoff is stable: the KV cache consumes memory to avoid repeatedly recomputing past attention states.